In [1]:
#Tsfnet per branch dropout
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\Preproc run f no beer lambert"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    """
    a_flat = a.reshape(-1)  # NEW
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    # def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
    # CHANGED (PER-BRANCH DROPOUT, per chat history): the three-way capacity comparison (full /
    # symmetric-reduced / asymmetric-reduced) showed EEG-only accuracy stayed statistically flat
    # (35.05% -> 35.01%) even when EEG's channel count was DOUBLED back (8->16 channels to 12->24),
    # while the shared dropout=0.6 and weight_decay=5e-2 were held constant across both experiments.
    # This is a strong signal that EEG's regression from its full-capacity baseline (37.43%) was
    # driven by the SHARED, uniformly-applied dropout/weight_decay increase, not by channel capacity
    # -- consistent with the multimodal-learning finding that different modalities overfit/underfit
    # at different rates and benefit from independently-tuned regularization (Wang, Tran & Feiszli,
    # 2020, "What Makes Training Multi-Modal Classification Networks Hard?", CVPR;
    # https://arxiv.org/abs/1905.12681). fNIRS, in contrast, IMPROVED under the same shared
    # dropout=0.6 (38.38% -> 40.11% in the asymmetric run) -- so a single shared dropout value
    # cannot be optimal for both branches simultaneously. Three independent dropout rates now
    # replace the single `dropout` parameter: eeg_dropout (lower, since EEG showed no benefit from
    # -- and was likely hurt by -- the stronger shared value), nirs_dropout (kept higher, since
    # fNIRS benefited), and fusion_dropout (for the EFGF/CAFE fusion mechanism and its own
    # classifier head, set to an intermediate default pending its own dedicated tuning).
    def __init__(self, eeg_ch=28, num_classes=3, eeg_dropout=0.5, nirs_dropout=0.65, fusion_dropout=0.55):  # CHANGED
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # EEG branch
        # CHANGED (ASYMMETRIC CAPACITY, per chat history): the previous SYMMETRIC reduction (8->16
        # for both branches) hurt EEG-only accuracy disproportionately (-2.38pp) while barely
        # affecting fNIRS (+0.20pp) or overall test accuracy (+0.07pp) -- consistent with EEG being
        # the richer, higher-information modality (200Hz vs 10Hz sampling) that likely needs more
        # capacity than fNIRS (which has been the consistently weaker branch across every run in
        # this project: A-F). EEG channels are now reduced LESS aggressively (12->24) than fNIRS.
        self.eeg_conv1 = nn.Conv3d(1, 12, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # CHANGED: 8 -> 12
        self.eeg_bn1 = nn.BatchNorm3d(12)  # CHANGED: 8 -> 12
        self.eeg_drop1 = nn.Dropout(eeg_dropout)  # CHANGED: uses eeg_dropout, not shared dropout
        self.eeg_conv2 = nn.Conv3d(12, 24, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # CHANGED: 8->12 in, 16->24 out
        self.eeg_bn2 = nn.BatchNorm3d(24)  # CHANGED: 16 -> 24
        self.eeg_drop2 = nn.Dropout(eeg_dropout)  # CHANGED: uses eeg_dropout, not shared dropout
        self.eeg_fc = nn.Linear(24 * 4 * 4, 64)  # CHANGED: 16*4*4=256 -> 24*4*4=384

        # fNIRS branch
        # CHANGED (ASYMMETRIC CAPACITY): fNIRS channels reduced MORE aggressively (6->12) than EEG,
        # since fNIRS has been the consistently weaker/lower-information branch across every run in
        # this project (A-F) -- lower sampling rate (10Hz), simpler hemodynamic signal.
        self.nirs_conv1 = nn.Conv3d(2, 6, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))  # CHANGED: 8 -> 6
        self.nirs_bn1 = nn.BatchNorm3d(6)  # CHANGED: 8 -> 6
        self.nirs_drop1 = nn.Dropout(nirs_dropout)  # CHANGED: uses nirs_dropout, not shared dropout
        self.nirs_conv2 = nn.Conv3d(6, 12, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))  # CHANGED: 8->6 in, 16->12 out
        self.nirs_bn2 = nn.BatchNorm3d(12)  # CHANGED: 16 -> 12
        self.nirs_drop2 = nn.Dropout(nirs_dropout)  # CHANGED: uses nirs_dropout, not shared dropout
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(12 * 4 * 4, 64)  # CHANGED: 16*4*4=256 -> 12*4*4=192

        # NEW (ASYMMETRIC CAPACITY support): since eeg_feat_map (24 channels) and nirs_feat_map (12
        # channels) NO LONGER share the same channel depth, the paper's Eq. 1 fusion arithmetic
        # (gamma*eeg_feat_map + (1-gamma)*nirs_feat_map + Phi*nirs_feat_map, a channel-wise sum)
        # cannot be applied directly -- a 1x1 Conv2d "channel-alignment" layer projects nirs_feat_map
        # up to EEG's channel depth (24) ONLY for use in this fusion equation. This is a standard,
        # well-established technique for combining feature maps of different channel depths in
        # multi-branch architectures (a 1x1 conv is a linear per-pixel channel re-projection, adding
        # negligible extra capacity: 12*24=288 additional weights). nirs_fc/nirs branch's OWN
        # classifier head still uses the native (unaligned) 12-channel nirs_feat_map, unaffected.
        self.nirs_channel_align = nn.Conv2d(12, 24, kernel_size=1)  # NEW: aligns nirs_feat_map's channel depth to EEG's (12 -> 24) for the fusion equation only

        # EFGF
        # CHANGED (ASYMMETRIC CAPACITY): each attention conv now reads its OWN branch's native
        # channel depth (24 for EEG, 12 for fNIRS) -- output is always 1 channel regardless, so no
        # alignment is needed here (only the Eq.1 fusion SUM below needs channel alignment).
        self.efgf_eeg_attn = nn.Conv2d(24, 1, kernel_size=1)   # CHANGED: 16 -> 24, matches EEG's native depth
        self.efgf_nirs_attn = nn.Conv2d(12, 1, kernel_size=1)  # CHANGED: 16 -> 12, matches fNIRS's native depth
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW
        self.fusion_proj = nn.Linear(24 * 4 * 4, 64)  # CHANGED: 16*4*4=256 -> 24*4*4=384 -- fusion representation uses EEG's (larger) channel depth as the common basis

        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        self.nirs_seq_proj = nn.Linear(12, 64)  # CHANGED: 16 -> 12, matches fNIRS's native (unaligned) channel depth used for the temporal CAFE sequence
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=fusion_dropout)  # CHANGED: uses fusion_dropout, not shared dropout
        self.cafe_gate = nn.Linear(128, 64)
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=fusion_dropout)  # CHANGED: uses fusion_dropout, not shared dropout
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        # self.dropout = nn.Dropout(dropout)
        # CHANGED (PER-BRANCH DROPOUT): the single shared self.dropout, previously applied to
        # eeg_feat, final_nirs_feat, AND final_feat identically before their respective classifier
        # heads, is replaced with three independent dropout modules matching each branch's own rate.
        self.eeg_out_drop = nn.Dropout(eeg_dropout)      # NEW: applied before fc_eeg
        self.nirs_out_drop = nn.Dropout(nirs_dropout)    # NEW: applied before fc_fnirs
        self.fusion_out_drop = nn.Dropout(fusion_dropout)  # NEW: applied before fc_fusion
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_map = y.mean(dim=2)  # NEW
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED

        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW

        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        # nirs_feat_map's native depth (12) is projected to EEG's depth (24) ONLY for this fusion sum.
        nirs_feat_map_aligned = self.nirs_channel_align(nirs_feat_map)  # NEW: (batch,12,4,4) -> (batch,24,4,4)
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map_aligned + Phi * nirs_feat_map_aligned  # CHANGED: uses nirs_feat_map_aligned instead of nirs_feat_map
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW

        nirs_seq = y_spatial.transpose(1, 2) 
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        nirs_query = nirs_feat.unsqueeze(1)   # NEW
        fused_kv = fused_feat.unsqueeze(1)    # NEW
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW

        eeg_out = self.fc_eeg(self.eeg_out_drop(eeg_feat))              # CHANGED: self.dropout -> self.eeg_out_drop
        fnirs_out = self.fc_fnirs(self.nirs_out_drop(final_nirs_feat))  # CHANGED: self.dropout -> self.nirs_out_drop
        fusion_out = self.fc_fusion(self.fusion_out_drop(final_feat))  # CHANGED: self.dropout -> self.fusion_out_drop
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        Phi_broadcast = Phi.expand(-1, 24, -1, -1)  # CHANGED: 16 -> 24, matches eeg_feat_map's new native channel depth
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.6).to(device)
    # CHANGED (PER-BRANCH DROPOUT): eeg_dropout lowered back to 0.5 (the original, since doubling
    # EEG's channels under the shared 0.6 rate did not recover EEG-only accuracy at all -- 35.05%
    # -> 35.01%, statistically flat -- pointing to the shared dropout, not channel count, as the
    # likely cause of EEG's regression). nirs_dropout kept at 0.65 (slightly above the previous
    # shared 0.6) since fNIRS-only accuracy IMPROVED under the stronger shared value in both prior
    # capacity-reduction runs. fusion_dropout set to an intermediate 0.55 pending its own tuning.
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, eeg_dropout=0.5, nirs_dropout=0.65, fusion_dropout=0.55).to(device)  # CHANGED  # CHANGED: dropout 0.5 -> 0.6, stronger regularization per chat history plan

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=5e-2)  # CHANGED: weight_decay 2e-2 -> 5e-2, stronger L2 regularization per chat history plan
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW
    criterion = nn.CrossEntropyLoss()  # NEW
    LAMBDA_EEG = 0.2     # CHANGED
    LAMBDA_FNIRS = 0.2   # CHANGED
    LAMBDA_EFGF_REG = 1.0   # NEW
    LAMBDA_CAFE_REG = 1.0   # NEW

    best_val_acc = 0.0
    epochs = 100

    EARLY_STOP_PATIENCE = 15  # NEW
    epochs_no_improve = 0  # NEW

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW

            L_pred = criterion(pred, lbl_b)          # NEW
            L_eeg = criterion(eeg_out, lbl_b)        # NEW
            L_fnirs = criterion(fnirs_out, lbl_b)    # NEW
            L_fusion = criterion(fusion_out, lbl_b)  # NEW
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion  # NEW

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)      # NEW
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe  # NEW
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW
        else:  # NEW
            epochs_no_improve += 1  # NEW
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED
    print("\nClassification Report (final fused prediction):")  # CHANGED
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED

    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [2]:
#Variance check 3seeds
# چک واریانس چند-seed: آیا تفاوت‌های ۰.۵-۱ واحد درصدی که بین اجراهای اخیر دیدیم،
# اثر واقعی تغییرات معماری/dropout بوده یا فقط نویز initialization؟
#
# منطق: دقیقاً همان معماری نامتقارن (بهترین پیکربندی فعلی: کانال EEG=12->24, fNIRS=6->12,
# dropout=0.6 مشترک) را با ۳ seed متفاوت آموزش می‌دهیم. splitِ سوژه‌ها (train/val/test) ثابت
# می‌ماند (چون به RANDOM_SEED اصلی وابسته است، نه به seed مدل) -- فقط مقداردهی اولیه‌ی وزن‌ها
# و ترتیب batch ها بین اجراها فرق می‌کند.
#
# بر اساس: Bouthillier et al. (2021), "Accounting for Variance in Machine Learning Benchmarks",
# MLSys 2021. https://arxiv.org/abs/2103.03098

import os
import gc
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# ============================== CONFIGURATION ==============================
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\Preproc run f no beer lambert"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# NEW: SPLIT_SEED فقط برای تقسیم سوژه‌ها استفاده می‌شود -- ثابت می‌ماند در همه‌ی اجراها، تا
# مقایسه‌ی seed ها فقط تفاوت "مقداردهی اولیه‌ی مدل" را بسنجد، نه تفاوت در اینکه کدام سوژه‌ها
# در train/val/test افتاده‌اند (که خودش یک منبع دیگر و بزرگ‌تر از واریانس است).
SPLIT_SEED = 42  # NEW
random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ============================== DATASET (verbatim) ==============================
class MMapHybridDataset(Dataset):
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0

        print(f"Scanning {dataset_name} - {len(subs_list)} subjects...")

        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)

            if not os.path.exists(sub_folder):
                continue

            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"),
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)

            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"),
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)

            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")

            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]

                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}).")
                        continue

                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")

                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue

        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name}!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"var_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"var_mmap_{dataset_name[:3]}_n.npy")

        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass

        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')

        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()

        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor

    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass


# ============================== MODEL (asymmetric-capacity architecture, verbatim) ==============================
def pearson_correlation_loss(a, b, eps=1e-8):
    a_flat = a.reshape(-1)
    b_flat = b.reshape(-1)
    a_centered = a_flat - a_flat.mean()
    b_centered = b_flat - b_flat.mean()
    numerator = (a_centered * b_centered).sum()
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps
    pcc = numerator / denominator
    return 1.0 - torch.abs(pcc)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))


class TSFNet_With_fNIRS_2D(nn.Module):
    # NOTE: this is the ASYMMETRIC-CAPACITY architecture (best config so far, Test=39.93%):
    # EEG channels 12->24, fNIRS channels 6->12, single shared dropout (matches the run being
    # variance-checked here -- NOT the per-branch-dropout variant, which underperformed).
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.6):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        self.eeg_conv1 = nn.Conv3d(1, 12, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(12)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(12, 24, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(24)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(24 * 4 * 4, 64)

        self.nirs_conv1 = nn.Conv3d(2, 6, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(6)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(6, 12, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(12)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)
        self.nirs_fc = nn.Linear(12 * 4 * 4, 64)
        self.nirs_channel_align = nn.Conv2d(12, 24, kernel_size=1)

        self.efgf_eeg_attn = nn.Conv2d(24, 1, kernel_size=1)
        self.efgf_nirs_attn = nn.Conv2d(12, 1, kernel_size=1)
        self.alpha = nn.Parameter(torch.tensor(0.0))
        self.gamma = nn.Parameter(torch.tensor(0.0))
        self.fusion_proj = nn.Linear(24 * 4 * 4, 64)

        self.pos_encoder = PositionalEncoding(64, max_len=100)
        self.nirs_seq_proj = nn.Linear(12, 64)
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate2 = nn.Linear(128, 64)

        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)

        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, nirs):
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1)
        nirs_feat_map = y.mean(dim=2)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))

        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn

        gamma = torch.clamp(self.gamma, 0, 1)
        nirs_feat_map_aligned = self.nirs_channel_align(nirs_feat_map)
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map_aligned + Phi * nirs_feat_map_aligned
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))

        nirs_seq = y_spatial.transpose(1, 2)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)
        nirs_proj = self.pos_encoder(nirs_seq_proj)

        fused_seq = fused_feat.unsqueeze(1)
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)

        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        nirs_query = nirs_feat.unsqueeze(1)
        fused_kv = fused_feat.unsqueeze(1)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out

        Phi_broadcast = Phi.expand(-1, 24, -1, -1)
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced


# ============================== LOAD DATA ONCE (shared across all seed runs) ==============================
print("Loading datasets once (shared across all seed runs)...")
train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

criterion = nn.CrossEntropyLoss()
LAMBDA_EEG = 0.2
LAMBDA_FNIRS = 0.2
LAMBDA_EFGF_REG = 1.0
LAMBDA_CAFE_REG = 1.0


def train_and_evaluate_one_seed(seed, max_epochs=100, early_stop_patience=15):
    """
    Trains the asymmetric-capacity TSFNet with a given model-init seed (NOT the subject-split
    seed, which stays fixed) and returns a dict of test-set metrics.
    """
    torch.manual_seed(seed)  # NEW: only this changes between runs -- controls weight init + dropout/batch-order randomness

    # DataLoaders recreated per-seed so shuffle order also varies with the seed (a legitimate,
    # intended source of the variance we are trying to measure -- see Bouthillier et al. 2021).
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.6).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=5e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    best_val_acc = 0.0
    epochs_no_improve = 0
    ckpt = os.path.join(PROCESSED_DATA_PATH, f'variance_check_seed{seed}.pth')

    for epoch in range(1, max_epochs + 1):
        model.train()
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred = criterion(pred, lbl_b)
            L_eeg = criterion(eeg_out, lbl_b)
            L_fnirs = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out, lbl_b)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)
            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f"  [seed {seed}] Epoch {epoch:3d}/{max_epochs} | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= early_stop_patience:
            print(f"  [seed {seed}] Early stopping at epoch {epoch}. Best Val: {best_val_acc*100:.2f}%")
            break

    # ---- Test evaluation ----
    model.load_state_dict(torch.load(ckpt))
    model.eval()
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    results = {
        'seed': seed,
        'best_val_acc': best_val_acc * 100,
        'test_acc': accuracy_score(all_true, all_preds_final) * 100,
        'eeg_acc': accuracy_score(all_true, all_preds_eeg) * 100,
        'fnirs_acc': accuracy_score(all_true, all_preds_fnirs) * 100,
        'fusion_acc': accuracy_score(all_true, all_preds_fusion) * 100,
    }

    del model
    gc.collect()
    torch.cuda.empty_cache()
    if os.path.exists(ckpt):
        os.remove(ckpt)

    return results


# ============================== RUN 3 SEEDS ==============================
if __name__ == "__main__":
    SEEDS = [42, 123, 2024]  # NEW: three different model-init seeds; subject split stays fixed
    all_results = []

    for seed in SEEDS:
        print(f"\n{'='*70}\nSTARTING RUN WITH SEED {seed}\n{'='*70}")
        result = train_and_evaluate_one_seed(seed)
        all_results.append(result)
        print(f"[seed {seed}] DONE -- Test: {result['test_acc']:.2f}% | "
              f"EEG: {result['eeg_acc']:.2f}% | fNIRS: {result['fnirs_acc']:.2f}% | "
              f"Fusion: {result['fusion_acc']:.2f}%")

    # ============================== SUMMARY: MEAN +/- STD ACROSS SEEDS ==============================
    print("\n" + "=" * 70)
    print("VARIANCE CHECK SUMMARY (3 seeds, same architecture/hyperparameters)")
    print("=" * 70)
    print(f"{'seed':<8}{'Val':>10}{'Test':>10}{'EEG':>10}{'fNIRS':>10}{'Fusion':>10}")
    for r in all_results:
        print(f"{r['seed']:<8}{r['best_val_acc']:>10.2f}{r['test_acc']:>10.2f}"
              f"{r['eeg_acc']:>10.2f}{r['fnirs_acc']:>10.2f}{r['fusion_acc']:>10.2f}")

    for metric in ['best_val_acc', 'test_acc', 'eeg_acc', 'fnirs_acc', 'fusion_acc']:
        vals = [r[metric] for r in all_results]
        mean_v = np.mean(vals)
        std_v = np.std(vals)
        print(f"\n{metric}: mean={mean_v:.2f}%  std={std_v:.2f}%  range=[{min(vals):.2f}, {max(vals):.2f}]")

    print("\n" + "=" * 70)
    print("تفسیر: اگر std هر معیار نزدیک یا بزرگ‌تر از تفاوت‌هایی بود که بین اجراهای قبلی")
    print("(کامل/متقارن/نامتقارن/dropout مستقل) دیدیم (۰.۲۵ تا ۰.۸۹ واحد درصد)، یعنی آن")
    print("تفاوت‌ها احتمالا در حد نویز initialization بوده‌اند، نه اثر واقعی تغییرات معماری.")
    print("=" * 70)

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed.")

Loading datasets once (shared across all seed runs)...
Scanning Train_Set - 18 subjects...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready! Total Windows: 3065 | Labels: [1023 1023 1019]

S